# Pipeline

Jusqu'à présent nous avons fait le pre processing et la validation séparément. Mais serait-il possible de rassembler le tout?

Oui, et c'est ce que feront les pipelines, elles vont nous permettre de regrouper les différents transformer utilisé ainsi que notre modèle dans une seule et même pipeline que nous pourrons ensuite utiliser pour appliquer directement les transformations nécéssaire à nos données et ensuite en faire les estimations.

Mais qu'ouis-je? Comment se fasse que je n'eu point entendu parler de tel prodiges par le passé?

Et bien rectifions la chose dès maintenant et parlons des pipelines!

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.datasets import load_iris

In [3]:
iris = load_iris()

X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5)

# Transformer
scaler = StandardScaler()
X_train_Transform = scaler.fit_transform(X_train)

# Estimator
model = SGDClassifier(random_state=0)
model.fit(X_train_Transform, y_train)

# Test
X_test_transform = scaler.transform(X_test)
model.score(X_test_transform, y_test)

0.9

Comment pouvons nous donc simplifier ce code avec une pipeline?

Nous allons maintenant regrouper notre transformer et notre model dans une pipeline ce qui nous retournera un estimateur composite.

Lors de l'utilisation de la méthode fit, il utilisera donc la méthod fit_transform des transformer et la méthode fit des model.

Et lors de l'utilisation de la méthode prédict, il va utiliser la méthode transform des transformer et predict de l'estimateur.

In [4]:
from sklearn.pipeline import make_pipeline

In [5]:
model = make_pipeline(StandardScaler(), SGDClassifier())

model.fit(X_train, y_train)

model.predict(X_test)

array([1, 1, 2, 0, 2, 0, 0, 2, 0, 2, 1, 1, 2, 2, 0, 0, 2, 2, 0, 0, 1, 2,
       0, 1, 1, 2, 1, 1, 1, 2])

In [6]:
model.score(X_test, y_test)

0.8333333333333334

La pipeline va nous apporter plusieurs avantages : 
- simplifier l'utilisation de nos transformer/estimateur
- Éviter d'avoir des données mal transformées
- Fera la cross validation automatiquement

L'avantage de la pipeline est aussi que l'on va pouvoir utiliser GridSearchCV pour définir les meilleurs paramètre de l'entièreté de la pipeline :

In [7]:
from sklearn.model_selection import GridSearchCV

In [8]:
model = make_pipeline(PolynomialFeatures(),
                      StandardScaler(),
                      SGDClassifier(random_state=0))

model

Pipeline(steps=[('polynomialfeatures', PolynomialFeatures()),
                ('standardscaler', StandardScaler()),
                ('sgdclassifier', SGDClassifier(random_state=0))])

L'utilisation de GridSearchCV sur une pipeline est un peu différente de sur un estimateur.

Ici on va passer un dictionnaire qui contiendra les différents paramètre pour chacune des étapes de notre pipeline.

Et la syntaxe de la clé de ce dictionnaire sera la suivante :
COMPOSANT__PARAM (!!! Il s'agit bien d'un double "_" !!!)

In [9]:
params = {
    'polynomialfeatures__degree' : [2,3,4],
    'sgdclassifier__penalty' : ['l1', 'l2']
}

grid = GridSearchCV(model, param_grid=params, cv=5)

grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('polynomialfeatures',
                                        PolynomialFeatures()),
                                       ('standardscaler', StandardScaler()),
                                       ('sgdclassifier',
                                        SGDClassifier(random_state=0))]),
             param_grid={'polynomialfeatures__degree': [2, 3, 4],
                         'sgdclassifier__penalty': ['l1', 'l2']})

In [10]:
grid.best_params_

{'polynomialfeatures__degree': 2, 'sgdclassifier__penalty': 'l1'}

In [11]:
grid.best_score_

0.975

In [12]:
estimator = grid.best_estimator_